# 01 · รวมและทำความสะอาดข้อมูล

เป้าหมาย: สร้าง 2 ไฟล์ใน `data/processed/` ตามที่กำหนดใน `data/README.md`

1. `hourly_khonkaen.csv` ตารางรายชั่วโมงสำหรับพยากรณ์ (สถานี Air4Thai + Open-Meteo)
2. `device_readings.csv` ค่าจากเครื่องเรา (Firebase `/history` หรือ SD card) หลังแก้ความชื้น

และประเมินเซนเซอร์โดยไม่ต้องตั้งเครื่องคู่สถานี: เทียบค่าเฉลี่ยรายวันกับสถานีและ CAMS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# หา root ของ repo (โฟลเดอร์ที่มี data/) ไม่ว่าจะเปิด notebook จากที่ไหน
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data").is_dir())
RAW, PROC = ROOT / "data" / "raw", ROOT / "data" / "processed"
PROC.mkdir(parents=True, exist_ok=True)

## 1. ค่าสถานี Air4Thai

ดาวน์โหลดรายชั่วโมงจากเว็บ Air4Thai ได้ครั้งละ 1 เดือน วางไฟล์ทั้งหมดไว้ใน `data/raw/air4thai/<stationID>/`
รูปแบบคอลัมน์ของไฟล์ที่ดาวน์โหลดอาจต่างกัน ให้แก้ `COLMAP` ให้ตรงกับไฟล์จริง

In [ ]:
COLMAP = {            # ชื่อคอลัมน์ในไฟล์ Air4Thai -> ชื่อที่ใช้ในโปรเจกต์
    "Date": "date", "Time": "hour",
    "PM2.5": "pm25", "PM10": "pm10", "O3": "o3", "CO": "co", "NO2": "no2", "SO2": "so2",
}

def load_air4thai(station_dir, prefix):
    files = sorted(Path(station_dir).glob("*.csv")) + sorted(Path(station_dir).glob("*.xlsx"))
    parts = []
    for f in files:
        d = pd.read_excel(f) if f.suffix == ".xlsx" else pd.read_csv(f)
        d = d.rename(columns=COLMAP)
        hour = d["hour"].astype(str).str.slice(0, 2).astype(int) if "hour" in d else 0
        d["time"] = pd.to_datetime(d["date"]) + pd.to_timedelta(hour, unit="h")
        keep = [c for c in ["pm25", "pm10", "o3", "co", "no2", "so2"] if c in d]
        d = d[["time"] + keep].apply(lambda s: pd.to_numeric(s, errors="coerce") if s.name != "time" else s)
        parts.append(d)
    if not parts:
        print("ไม่พบไฟล์ใน", station_dir)
        return pd.DataFrame(columns=["time"])
    df = pd.concat(parts).drop_duplicates("time").set_index("time").sort_index()
    df = df.mask(df < 0)                       # Air4Thai ใช้ค่าติดลบแทนข้อมูลหาย
    return df.add_prefix(prefix)

kk = load_air4thai(RAW / "air4thai" / "KK_STATION_ID", "station_")   # แก้เป็นรหัสสถานีขอนแก่น
mk = load_air4thai(RAW / "air4thai" / "MK_STATION_ID", "mk_")        # แก้เป็นรหัสสถานีมหาสารคาม
kk.describe()

## 2. Open-Meteo (ERA5 + CAMS)

สร้างไฟล์ด้วย `python scripts/fetch_open_meteo.py --start ... --end ...`

In [ ]:
met_files = sorted(RAW.glob("open_meteo_khonkaen_*.csv"))
met = pd.concat([pd.read_csv(f, parse_dates=["time"]) for f in met_files]).drop_duplicates("time").set_index("time").sort_index() if met_files else pd.DataFrame()
met.head()

## 3. รวมเป็นตารางรายชั่วโมง

In [ ]:
hourly = kk.join(mk, how="outer").join(met, how="left")
hourly = hourly.asfreq("h")                                   # ช่องว่างเป็น NaN ชัดเจน
print("ช่วงข้อมูล:", hourly.index.min(), "ถึง", hourly.index.max())
print("สัดส่วนข้อมูลหาย:\n", hourly.isna().mean().round(3).sort_values(ascending=False).head(10))
hourly.to_csv(PROC / "hourly_khonkaen.csv")

## 4. ค่าจากเครื่องเรา

Export ได้จาก Firebase Console > Realtime Database > ⋮ > Export JSON แล้ววางเป็น `data/raw/firebase_export.json`
หรือใช้ไฟล์ CSV จาก SD card (`data/raw/sd/*.csv`)

**การแก้ความชื้นแบบ EPA (Barkjohn et al., 2021)** สำหรับ PMS5003 ค่า CF=1 (ถ้ายังไม่มี BME280 จะใช้ความชื้นจาก ERA5 แทน):
`PM2.5_corr = 0.524 × PM2.5_cf1 − 0.0862 × RH + 5.75`
ตรวจก่อนว่าไลบรารีที่ใช้อ่าน PMS5003 ส่งค่า CF=1 หรือ ATM

In [ ]:
import json

def load_firebase_history(path):
    data = json.loads(Path(path).read_text(encoding="utf-8"))
    hist = data.get("history", data)                     # รองรับทั้ง export ทั้งก้อนและเฉพาะ /history
    df = pd.DataFrame.from_dict(hist, orient="index")
    df["time"] = pd.to_datetime(df["timestamp"], unit="ms", utc=True).dt.tz_convert("Asia/Bangkok").dt.tz_localize(None)
    return df.set_index("time").sort_index()

def barkjohn(pm25_cf1, rh):
    return (0.524 * pm25_cf1 - 0.0862 * rh + 5.75).clip(lower=0)

fx = RAW / "firebase_export.json"
dev = load_firebase_history(fx) if fx.exists() else pd.DataFrame()
if not dev.empty:
    cf1 = dev["pm2_5_cf1"] if "pm2_5_cf1" in dev else dev["pm2_5"]   # สูตร EPA ใช้ค่า CF=1
    if "humidity" in dev and dev["humidity"].notna().any():
        rh = dev["humidity"]
    else:                                                       # ยังไม่มี BME280: ใช้ RH รายชั่วโมงจาก ERA5 แทน
        rh = hourly["relative_humidity_2m"].reindex(dev.index.floor("h")).to_numpy()
        print("ไม่มีความชื้นจากเครื่อง ใช้ relative_humidity_2m จาก Open-Meteo แทน")
    dev["pm25_corr"] = barkjohn(cf1, rh)
    if "ens_status" in dev:                                      # ตัดค่าก๊าซช่วงอุ่นเครื่องของ ENS160
        dev.loc[dev["ens_status"] != 0, ["tvoc", "eco2", "aqi"]] = np.nan
    dev.to_csv(PROC / "device_readings.csv")
dev.head()

## 5. ประเมินเซนเซอร์: ค่าเฉลี่ยรายวันเทียบสถานีและ CAMS

In [ ]:
if not dev.empty:
    daily = pd.DataFrame({
        "device": dev["pm25_corr"].resample("D").mean(),
        "station": hourly["station_pm25"].resample("D").mean(),
        "cams": hourly.get("cams_pm2_5", pd.Series(dtype=float)).resample("D").mean(),
    }).dropna(subset=["device", "station"])
    err = daily["device"] - daily["station"]
    print(f"n = {len(daily)} วัน  r = {daily['device'].corr(daily['station']):.2f}  "
          f"MAE = {err.abs().mean():.1f}  bias = {err.mean():+.1f} µg/m³")
    daily.plot(marker="o", figsize=(10, 4), title="PM2.5 เฉลี่ยรายวัน")
    plt.ylabel("µg/m³"); plt.show()